In [11]:
import pandas as pd
import numpy as np

from statsmodels.stats.multitest import fdrcorrection
from scipy.stats import chi2, mannwhitneyu
from timeit import default_timer as timer
from joblib import Parallel, delayed
from tqdm import tqdm
import warnings
from sklearn.exceptions import ConvergenceWarning



In [3]:
prodict_metadata = pd.read_excel("/home/lestrada/projects/ProDICT/data/_cohort_files/METADATA_PANCANCER_PAPER_260416.xlsx")

In [62]:
new_metadata = pd.read_excel("/home/lestrada/projects/ProDICT/data/_cohort_files/METADATA_PANCANCER_PAPER_260416.xlsx")

In [7]:
#display(prodict_metadata[['Sample name', 'code_oncotree']].head())
display(new_metadata[['Sample name', 'code_oncotree', 'oncotree_code_enrollment', 'PROdict']].head())

,Sample name,code_oncotree,oncotree_code_enrollment,PROdict
0,H021-9ZENZBK-M3-Q1,AMPCA,AMPCA,TCC low or NA
1,H021-7YTCA83-M1-Q1,APAD,APAD,TRAIN
2,H021-7QUJ2X-M2-Q1,APAD,APAD,TEST
3,K26K-VNZ1KD-M11-Q1,BRCA,BRCA,TRAIN
4,K26K-A927U9-M22-Q1,BRCA,BRCA,TRAIN


In [15]:
prodict_metadata['Sample name'] = prodict_metadata['Sample name'].str.strip()
new_metadata['Sample name'] = new_metadata['Sample name'].str.strip()

In [16]:
comparison_df = prodict_metadata[['Sample name', 'code_oncotree']].merge(new_metadata[['Sample name', 'oncotree_code', 'oncotree_code_enrollment']], how="outer", on='Sample name')

In [17]:
comparison_df.head()

,Sample name,code_oncotree,oncotree_code,oncotree_code_enrollment
0,A26K-5SXQR3-T11-Q1,BRCA,BRCA,BRCA
1,A26K-5SXQR3-T24-Q1,BRCA,BRCA,BRCA
2,A26K-9TET1N-T11-Q1,BRCA,BRCA,BRCA
3,A26K-ADUQXR-T11-Q1,BRCA,BRCA,BRCA
4,A26K-HS3BDB-T14-Q1,BRCA,BRCA,BRCA


In [18]:
comparison_df['code_oncotree'] == comparison_df['oncotree_code']

0       True
1       True
2       True
3       True
4       True
        ... 
1993    True
1994    True
1995    True
1996    True
1997    True
Length: 1998, dtype: bool

In [20]:
comparison_df[~(comparison_df['code_oncotree'] == comparison_df['oncotree_code'])].to_excel("/home/lestrada/projects/ProDICT/data/_cohort_files/oncotree_code_updates.xlsx", index=False)

In [24]:
max(prodict_metadata['Batch_No'])
min(prodict_metadata['Batch_No'])

1

In [25]:
new_metadata

,Sample name,Patient_Identifier,PID,Batch_No,TMT_Channel,Program,QC,QC_PP_(intensity_ratio_ref),Histology,Tissue_of_origin,...,ICDO3_topo_version,oncotree_code_onkostar,oncotree_code_enrollment,MTBslide_filename.1,MTBslide_all_filenames.1,MTBslide_patient_id.1,MTBslide_date.1,MTBslide_match_type,incidence_per_100K,incidence_reference
0,H021-9ZENZBK-M3-Q1,9ZENZBK,P2539,253,9,MASTER,passed,passed,Adenocarcinoma,AMPULLA_OF_VATER,...,rev2de,AMPULLA_OF_VATER,AMPCA,H021-9ZENZBK-M3-Q1_proteomics_241111.pptx,H021-9ZENZBK-M3-Q1_proteomics_241111.pptx,H021-9ZENZBK-M3-Q1,2024-11-11,exact,0.6,SEER
1,H021-7YTCA83-M1-Q1,7YTCA83,P3148,314,8,MASTER,passed,passed,Adenocarcinoma,BOWEL,...,rev2de,APAD,APAD,H021-7YTCA83-M1-Q1_proteomics_250602.pptx,H021-7YTCA83-M1-Q1_proteomics_250602.pptx,H021-7YTCA83-M1-Q1,2025-06-02,exact,1.1,SEER
2,H021-7QUJ2X-M2-Q1,7QUJ2X,P1864,186,4,MASTER,passed,passed,Adenocarcinoma,BOWEL,...,rev2de,APAD,APAD,H021-7QUJ2X-M2-Q1_proteomics_240331.pptx,H021-7QUJ2X-M2-Q1_proteomics_240331.pptx,H021-7QUJ2X-M2-Q1,2024-03-31,exact,1.1,SEER
3,K26K-VNZ1KD-M11-Q1,VNZ1KD,P1647,164,7,CATCH,passed,failed,Adenocarcinoma,BREAST,...,NaN,NaN,BRCA,NaN,NaN,NaN,NaT,NaN,131,SEER
4,K26K-A927U9-M22-Q1,A927U9,P1707,170,7,CATCH,passed,passed,Adenocarcinoma,BREAST,...,NaN,NaN,BRCA,NaN,NaN,NaN,NaT,NaN,131,SEER
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1993,H021-XP4W4J8-M1-Q1,XP4W4J8,P3037,303,7,MASTER,passed,passed,Squamous cell carcinoma,BLADDER,...,rev2de,BLSC,BLSC,H021-XP4W4J8-M1-Q1_proteomics_results_250522.pptx,H021-XP4W4J8-M1-Q1_proteomics_results_250522.pptx,H021-XP4W4J8-M1-Q1,2025-05-22,exact,3-10 (male-female),2-5% of bladder cancer
1994,H021-QYVBXUN-M1-Q1,QYVBXUN,P3271,327,1,MASTER,passed,passed,Squamous cell carcinoma,VULVA,...,rev2de,VULVA,VSC,H021-QYVBXUN-M1-Q1_proteomics_250703.pptx,H021-QYVBXUN-M1-Q1_proteomics_250703.pptx,H021-QYVBXUN-M1-Q1,2025-07-03,exact,NaN,rare subtype of SCC
1995,H021-HGR7VDP-M4-E3,HGR7VDP,P3302,330,2,MASTER,passed,passed,Squamous cell carcinoma,OTHER,...,rev2de,SNSC,SCCNOS,H021-HGR7VDP-M4-E3_proteomics_250712.pptx,H021-HGR7VDP-M4-E3_proteomics_250712.pptx,H021-HGR7VDP-M4-E3,2025-07-12,exact,NaN,no exact numbers
1996,H021-LQCK3RB-T1-E2,LQCK3RB,P3345,334,5,MASTER,passed,passed,Squamous cell carcinoma,HEAD_NECK,...,rev2de,OCSC,OCSC,H021-LQCK3RB-T1-E2_proteomics_results_250722.pptx,H021-LQCK3RB-T1-E2_proteomics_results_250722.pptx,H021-LQCK3RB-T1-E2,2025-07-22,exact,NaN,"very rare, no exact number"


In [26]:
prodict_metadata

,Unnamed: 0,Sample name,Batch_No,TMT_Channel,Program,Patient_Identifier,QC,QC_PP_(intensity_ratio_ref),MULTIPLE samples_passed,code_oncotree,...,PP sum ratio to ref,FP between_batch_correction_factor,PP between_batch_correction_factor,FP in_batch_correction_factor,PP in_batch_correction_factor,Paper_extv2,QC_old,Multiple samples_info,PID,PROdict
0,0,A26K-5SXQR3-T11-Q1,141,5,CATCH,5SXQR3,passed,passed,MULTIPLE,BRCA,...,0.389946,3.589,0.6741,0.9772,0.9965,no,passed,missing,P1415,NaN
1,1,A26K-5SXQR3-T24-Q1,141,6,CATCH,5SXQR3,passed,passed,MULTIPLE,BRCA,...,0.326845,3.589,0.6741,1.247,1.225,yes,passed,missing,P1416,NaN
2,2,A26K-9TET1N-T11-Q1,180,2,CATCH,9TET1N,passed,passed,EXTRACTION REPLICATE,BRCA,...,0.859303,0.7753,0.3517,0.9904,0.8775,yes,passed,missing,P1802,NaN
3,3,A26K-ADUQXR-T11-Q1,145,1,CATCH,ADUQXR,passed,failed,MULTIPLE,BRCA,...,0.181131,3.691,0.4976,1.705,1.914,yes,passed,missing,P1451,NaN
4,4,A26K-HS3BDB-T14-Q1,99,1,CATCH,HS3BDB4,passed,passed,NONE,BRCA,...,0.571615,2.563,1.041,0.9305,0.8685,yes,passed,missing,P0991,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1993,1993,S033-33+029-T1-Q1,141,2,MASTER,S033-33+029,passed,failed,NONE,CHDM,...,0.180888,3.589,0.6741,2.123,2.981,yes,passed,missing,P1412,NaN
1994,1994,S033-33+030-T1-Q1,142,7,MASTER,S033-33+030,passed,passed,NONE,CHDM,...,0.436816,2.546,0.4129,1.161,1.266,yes,passed,missing,P1427,NaN
1995,1995,S033-33+035-T1-Q1,157,7,MASTER,S033-33+035,passed,passed,NONE,CHDM,...,0.70917,0.7893,1.691,0.7949,0.7285,yes,passed,missing,P1577,NaN
1996,1996,S033-33+036-T1-Q1,157,8,MASTER,S033-33+036,passed,passed,NONE,CHDM,...,0.393267,0.7893,1.691,1.566,1.511,no,passed,missing,P1578,NaN


In [9]:
new_metadata['code_oncotree'].str.endswith("NOS").sum()

np.int64(204)

### PRODICT column redoing

In [50]:
new_metadata = pd.read_excel("/media/kusterlab/internal_projects/active/TOPAS/Publications/WP3_MTBpaper_CancerCell/Figure 2/METADATA_PANCANCER_PAPER_260416.xlsx")

In [52]:
new_metadata['PROdict']

0               TRAIN
1                TEST
2               TRAIN
3               TRAIN
4                TEST
            ...      
1993    TCC low or NA
1994    TCC low or NA
1995              NOS
1996            TRAIN
1997             TEST
Name: PROdict, Length: 1998, dtype: object

In [ ]:
ho = pd.read_excel("/home/lestrada/projects/ProDICT/data/_cohort_files/held_out_df.xlsx", usecols='Sample name')

In [20]:
train = pd.read_excel("/home/lestrada/projects/ProDICT/data/_cohort_files/initial_training_df.xlsx", usecols='A')
nos = pd.read_excel("/home/lestrada/projects/ProDICT/data/_cohort_files/removed_samples_CUPNOS_NECNOS_NETNOS_SARCNOS_RCSNOS_SCCNOS_missing.xlsx", usecols='A')
low_tcc = pd.read_excel("/home/lestrada/projects/ProDICT/data/_cohort_files/removed_samples_very low_notdefined.xlsx", usecols='A')

In [42]:
new_metadata[new_metadata['Sample name'].isin(ho['Sample name'])]['PROdict'].value_counts()

PROdict
TEST    467
Name: count, dtype: int64

In [54]:
new_metadata['PROdict']

0         TRAIN
1          TEST
2         TRAIN
3         TRAIN
4         TRAIN
         ...   
1993    low_tcc
1994    low_tcc
1995        NOS
1996      TRAIN
1997      TRAIN
Name: PROdict, Length: 1998, dtype: object

In [53]:
new_metadata['PROdict'] = pd.NA

new_metadata.loc[
    new_metadata['Sample name'].isin(train['Sample name']),
    'PROdict'
] = "TRAIN"

new_metadata.loc[
    new_metadata['Sample name'].isin(nos['Sample name']),
    'PROdict'
] = "NOS"

new_metadata.loc[
    new_metadata['Sample name'].isin(low_tcc['Sample name']),
    'PROdict'
] = "low_tcc"

new_metadata.loc[
    new_metadata['Sample name'].isin(ho['Sample name']),
    'PROdict'
] = "TEST"

In [59]:
set(train['Sample name']) & set(ho['Sample name']) #& set(low_tcc['Sample name']) & set(ho['Sample name'])

set()

In [60]:
new_metadata.to_excel("/media/kusterlab/internal_projects/active/TOPAS/Publications/WP3_MTBpaper_CancerCell/Figure 2/METADATA_PANCANCER_PAPER_260416.xlsx", index=False)

### Sample number

In [74]:
sample_n = pd.DataFrame(new_metadata['code_oncotree'].value_counts())

In [84]:
sample_n[sample_n['count']>9]

,count
code_oncotree,
BRCA,238
CUPNOS,109
CHDM,102
SYNS,84
LMS,78
SARCNOS,64
ACYC,61
SFT,52
MFH,52


In [96]:
sample_n[sample_n.index =='ULMS']

,count
code_oncotree,
ULMS,25


## P-value correction

In [81]:
def statistic_from_coefficients (Coefficients_df:pd.DataFrame,
                                 verbose: bool = True):
    """
    Args:
        Coefficients_df: DataFrame that contains all the coefficients of the cross folded Logistic Regression.

    Returns:
        coefficients_stats: Dataframe with the mean, std, Coeficient of Variation and p_value of the coefficients. Values per each protein. .
        significant_proteins: List of significant proteins, sorted by coefficient value.
    """
    #Reading file and obtaining statistics from coefficients
    coefficients_stats = Coefficients_df.describe()
    coefficients_stats = coefficients_stats.T[coefficients_stats.loc['mean']!=0].T[1:3] #This selecst coffiecients different than 0 and the rows 'mean and sd'

    #Calculating frequency
    coefficients_stats.loc['Freq'] = [(Coefficients_df[column] != 0).sum()/ Coefficients_df[column].count() for column in coefficients_stats.columns]#-> Selecting by index

    # Calculating Wald Test for each coefficient
    coefficients_stats.loc['Wald Chi_Square'] = (np.square(coefficients_stats.loc['mean']))/(np.square(coefficients_stats.loc['std']))
    p_values = 1 - chi2.cdf(coefficients_stats.loc['Wald Chi_Square'], df=1)

    significant_proteins, p_corrected_proteins = fdrcorrection(p_values[:-5], alpha=.01, method='indep', is_sorted=False)
    significant_scores, p_corrected_scores = fdrcorrection(p_values[-5:], alpha=.01, method='indep', is_sorted=False)

    p_corrected = np.concatenate([p_corrected_proteins, p_corrected_scores])
    significant = np.concatenate([significant_proteins, significant_scores])

    coefficients_stats.loc['p_corrected'] = p_corrected
    coefficients_stats.loc['Significant'] = significant #Significance is definded with alpha = 0.01 from chi2 dist.


    #Defining if the Wald value is greater than the significance level at 99% = 6.635. N
    #coefficients_stats.loc['Significant'] = [True if i > 6.635 else False for i in coefficients_stats.loc['Wald Chi-Square'] ]

    coefficients_stats= coefficients_stats.transpose()

    #returning list of significant proteins in order of coefficient value
    protein_coefficients_stats = coefficients_stats.iloc[:-5,:].sort_values(by='mean', ascending=False) #Sort by mean value of coefficients

    significant_proteins = protein_coefficients_stats[protein_coefficients_stats['Significant']==1].index.tolist()
    mcc_mean = coefficients_stats.loc['MCC_score', 'mean'] if 'MCC_score' in coefficients_stats.index else pd.NA
    mcc_std = coefficients_stats.loc['MCC_score', 'std'] if 'MCC_score' in coefficients_stats.index else pd.NA


    ## Message to user ##
    if verbose:
        print("A total of", protein_coefficients_stats.shape[0], " had a coefficient different than zero:")
        if pd.isna(mcc_mean) or pd.isna(mcc_std):
            print("• Mean MCC score: NA")
        else:
            print("• Mean MCC score:",np.round(mcc_mean, 4), '±', np.round(mcc_std, 4))
        print()

        print("--"*20)
        print("• Top 3 proteins with highest coefficients:")
        print(protein_coefficients_stats.head(3))
        print()

        print("--"*20)
        print("• List of significant proteins:",significant_proteins)
        print(f"• Number of significant proteins: {len(significant_proteins)}")
        print()

        print("--"*20)
        if pd.isna(mcc_mean):
            print("• Mean MCC score is not available for this combination.")
        elif np.round(mcc_mean, 4) < 0.70:
            print(" ✖ Warning! ✖: The mean MCC score is below 0.7, indicating poor model performance.")
        elif 0.80 > np.round(mcc_mean, 4) > 0.70:
            print("✦ The mean MCC score is above 0.7, indicating good model performance. ✦")
        else:
            print("★ The mean MCC score is above 0.8, indicating reliable model performance. ★")

    return coefficients_stats, significant_proteins

In [2]:
chdm = pd.read_excel('/home/lestrada/projects/ProDICT/data/01_generated_models/03_01/CHDM_260507_results/CHDM_features_coefficients.xlsx')

In [80]:
chdm['MCC_score'].to_numpy()[:-1]

array([0.97456231, 0.97566545, 0.97566545, 0.97566545, 0.97456231,
       0.97566545, 0.92638942, 0.97456231, 1.        , 1.        ,
       0.97456231, 0.95179063, 0.9496337 , 0.97456231, 1.        ,
       0.97566545, 1.        , 0.95084677, 1.        , 0.9496337 ,
       0.97566545, 1.        , 0.94858729, 0.97667123, 0.97566545,
       0.94858729, 1.        , 1.        , 0.9496337 , 0.97566545,
       1.        , 0.97456231, 0.95179063, 0.97456231, 0.97456231,
       0.97667123, 0.9496337 , 0.9496337 , 0.97667123, 0.97566545,
       0.97456231, 0.97566545, 1.        , 0.97456231, 0.95179063,
       0.97456231, 0.97456231, 0.95478592, 0.92302295, 1.        ,
       1.        , 0.97566545, 0.97456231, 0.97566545, 1.        ,
       0.92302295, 1.        , 0.9496337 , 1.        , 0.97566545,
       0.95290438, 0.97456231, 0.97566545, 0.9496337 , 1.        ,
       0.97566545, 0.92638942, 0.97456231, 1.        , 0.97566545,
       0.97456231, 0.95179063, 0.97456231, 0.97566545, 0.97566

In [56]:
thyc = pd.read_excel('/home/lestrada/projects/ProDICT/data/BRCA_260512_results/BRCA_features_coefficients.xlsx')

In [82]:
chdm_stats, chdm_proteins = statistic_from_coefficients(chdm)

A total of 541  had a coefficient different than zero:
• Mean MCC score: 0.9731 ± 0.0208

----------------------------------------
• Top 3 proteins with highest coefficients:
             mean       std  Freq  Wald Chi_Square   p_corrected  Significant
AKR1B10  0.282653  0.046335   1.0        37.213075  2.864631e-07          1.0
SCARA5   0.243855  0.115341   1.0         4.469895  9.349253e-01          0.0
OLFML2A  0.233755  0.035748   1.0        42.759005  3.349663e-08          1.0

----------------------------------------
• List of significant proteins: ['AKR1B10', 'OLFML2A', 'FN1', 'SUSD5', 'RAB3B', 'ASAP2', 'GALNT3']
• Number of significant proteins: 7

----------------------------------------
★ The mean MCC score is above 0.8, indicating reliable model performance. ★


In [83]:
chdm_stats_x, chdm_proteins_x = statistic_from_coefficients(chdm.iloc[:,:-5])

A total of 536  had a coefficient different than zero:
• Mean MCC score: NA

----------------------------------------
• Top 3 proteins with highest coefficients:
             mean       std  Freq  Wald Chi_Square   p_corrected  Significant
AKR1B10  0.282653  0.046335   1.0        37.213075  2.838155e-07          1.0
SCARA5   0.243855  0.115341   1.0         4.469895  9.349253e-01          0.0
OLFML2A  0.233755  0.035748   1.0        42.759005  3.318705e-08          1.0

----------------------------------------
• List of significant proteins: ['AKR1B10', 'OLFML2A', 'FN1', 'SUSD5', 'RAB3B', 'ASAP2', 'GALNT3']
• Number of significant proteins: 7

----------------------------------------
• Mean MCC score is not available for this combination.


In [57]:
thyc_stats, thyc_proteins = statistic_from_coefficients(thyc)

A total of 4685  had a coefficient different than zero:
• Mean MCC score: 0.8565 ± 0.0334

----------------------------------------
• Top 3 proteins with highest coefficients:
             mean       std  Freq  Wald Chi_Square   p_corrected  Significant
TRPS1    0.422783  0.046089   1.0        84.147630  0.000000e+00          1.0
ELAPOR1  0.377705  0.041086   1.0        84.511357  0.000000e+00          1.0
CA12     0.249580  0.037004   1.0        45.489376  8.997147e-09          1.0

----------------------------------------
• List of significant proteins: ['TRPS1', 'ELAPOR1', 'CA12', 'LDHD', 'AP1M2', 'IRF6', 'SPINT2', 'CRTC2', 'MYO5B', 'HOOK1', 'SIX4', 'PZP', 'SIGLEC1', 'ZNHIT2', 'MID1']
• Number of significant proteins: 15

----------------------------------------
★ The mean MCC score is above 0.8, indicating reliable model performance. ★


In [58]:
thyc_stats_x, thyc_proteins_x = statistic_from_coefficients(thyc.iloc[:,:-5])

A total of 4680  had a coefficient different than zero:
• Mean MCC score: NA

----------------------------------------
• Top 3 proteins with highest coefficients:
             mean       std  Freq  Wald Chi_Square   p_corrected  Significant
TRPS1    0.422783  0.046089   1.0        84.147630  0.000000e+00          1.0
ELAPOR1  0.377705  0.041086   1.0        84.511357  0.000000e+00          1.0
CA12     0.249580  0.037004   1.0        45.489376  2.396681e-08          1.0

----------------------------------------
• List of significant proteins: ['TRPS1', 'ELAPOR1', 'CA12', 'LDHD', 'AP1M2', 'IRF6', 'SPINT2', 'CRTC2', 'MYO5B', 'HOOK1', 'PZP', 'SIGLEC1', 'ZNHIT2', 'MID1']
• Number of significant proteins: 14

----------------------------------------
• Mean MCC score is not available for this combination.


In [55]:
chdm_stats.sort_values(by=" p_value_corrected",ascending=True).head(15)

,mean,std,Freq,Wald Chi_Square,p_value_corrected,Significant
MCC_score,0.973070,0.020786,1.0,2191.460359,0.000000e+00,1.0
F1_0,0.998507,0.001147,1.0,757323.861671,0.000000e+00,1.0
F1_1,0.974309,0.019781,1.0,2426.058436,0.000000e+00,1.0
F1_weighted,0.997167,0.002174,1.0,210318.103851,0.000000e+00,1.0
OLFML2A,0.233755,0.035748,1.0,42.759005,6.761242e-09,1.0
AKR1B10,0.282653,0.046335,1.0,37.213075,9.637020e-08,1.0
RAB3B,0.181576,0.030766,1.0,34.831418,2.804291e-07,1.0
Intercept,-3.136720,0.592000,1.0,28.074216,7.968258e-06,1.0
SUSD5,0.195008,0.042565,1.0,20.989336,2.801871e-04,1.0
GALNT3,0.173006,0.039469,1.0,19.213772,5.800863e-04,1.0


In [85]:
chdm_stats.sort_values(by="p_corrected",ascending=True).head(15)

,mean,std,Freq,Wald Chi_Square,p_corrected,Significant
MCC_score,0.973070,0.020786,1.0,2191.460359,0.000000e+00,1.0
F1_0,0.998507,0.001147,1.0,757323.861671,0.000000e+00,1.0
F1_1,0.974309,0.019781,1.0,2426.058436,0.000000e+00,1.0
F1_weighted,0.997167,0.002174,1.0,210318.103851,0.000000e+00,1.0
OLFML2A,0.233755,0.035748,1.0,42.759005,3.349663e-08,1.0
Intercept,-3.136720,0.592000,1.0,28.074216,1.167510e-07,1.0
AKR1B10,0.282653,0.046335,1.0,37.213075,2.864631e-07,1.0
RAB3B,0.181576,0.030766,1.0,34.831418,6.483426e-07,1.0
SUSD5,0.195008,0.042565,1.0,20.989336,6.246480e-04,1.0
FN1,0.211846,0.048184,1.0,19.330355,1.053753e-03,1.0


In [54]:
chdm_stats_x.sort_values(' p_value_corrected',ascending=True).head(10)

,mean,std,Freq,Wald Chi_Square,p_value_corrected,Significant
OLFML2A,0.233755,0.035748,1.0,42.759005,3.349663e-08,1.0
AKR1B10,0.282653,0.046335,1.0,37.213075,2.864631e-07,1.0
RAB3B,0.181576,0.030766,1.0,34.831418,6.483426e-07,1.0
SUSD5,0.195008,0.042565,1.0,20.989336,6.246480e-04,1.0
FN1,0.211846,0.048184,1.0,19.330355,1.053753e-03,1.0
GALNT3,0.173006,0.039469,1.0,19.213772,1.053753e-03,1.0
ASAP2,0.177398,0.041699,1.0,18.099005,1.620766e-03,1.0
EHD3,0.121831,0.032888,1.0,13.722600,1.432901e-02,0.0
CD109,0.101696,0.029968,1.0,11.515436,4.148907e-02,0.0
ABCG2,0.181544,0.055278,1.0,10.786126,5.532466e-02,0.0


In [87]:
chdm_stats_x.sort_values('p_corrected',ascending=True).head(10)

,mean,std,Freq,Wald Chi_Square,p_corrected,Significant
OLFML2A,0.233755,0.035748,1.0,42.759005,3.318705e-08,1.0
AKR1B10,0.282653,0.046335,1.0,37.213075,2.838155e-07,1.0
RAB3B,0.181576,0.030766,1.0,34.831418,6.423505e-07,1.0
SUSD5,0.195008,0.042565,1.0,20.989336,6.188749e-04,1.0
FN1,0.211846,0.048184,1.0,19.330355,1.044014e-03,1.0
GALNT3,0.173006,0.039469,1.0,19.213772,1.044014e-03,1.0
ASAP2,0.177398,0.041699,1.0,18.099005,1.605787e-03,1.0
EHD3,0.121831,0.032888,1.0,13.722600,1.419658e-02,0.0
CD109,0.101696,0.029968,1.0,11.515436,4.110562e-02,0.0
ABCG2,0.181544,0.055278,1.0,10.786126,5.481334e-02,0.0


In [65]:
thyc_stats.sort_values(by="p_corrected",ascending=True).head(22)

,mean,std,Freq,Wald Chi_Square,p_corrected,Significant
MCC_score,0.856527,0.033420,1.0,656.870630,0.000000e+00,1.0
F1_0,0.979119,0.005224,1.0,35126.644500,0.000000e+00,1.0
F1_1,0.875014,0.029141,1.0,901.640478,0.000000e+00,1.0
Intercept,-0.824896,0.048766,1.0,286.129870,0.000000e+00,1.0
TRPS1,0.422783,0.046089,1.0,84.147630,0.000000e+00,1.0
ELAPOR1,0.377705,0.041086,1.0,84.511357,0.000000e+00,1.0
F1_weighted,0.965058,0.008422,1.0,13130.939923,0.000000e+00,1.0
CA12,0.249580,0.037004,1.0,45.489376,8.997147e-09,1.0
LDHD,0.232498,0.040203,1.0,33.444643,3.820862e-06,1.0
IRF6,0.214849,0.037404,1.0,32.994057,4.335483e-06,1.0


In [63]:
thyc_stats_x.sort_values(by="p_corrected",ascending=True).head(15)

,mean,std,Freq,Wald Chi_Square,p_corrected,Significant
ELAPOR1,0.377705,0.041086,1.0,84.511357,0.000000e+00,1.0
TRPS1,0.422783,0.046089,1.0,84.147630,0.000000e+00,1.0
CA12,0.249580,0.037004,1.0,45.489376,2.396681e-08,1.0
LDHD,0.232498,0.040203,1.0,33.444643,8.587775e-06,1.0
IRF6,0.214849,0.037404,1.0,32.994057,8.661723e-06,1.0
SPINT2,0.188737,0.033414,1.0,31.905241,1.264011e-05,1.0
MYO5B,0.178804,0.033049,1.0,29.270648,4.212613e-05,1.0
AP1M2,0.228190,0.047113,1.0,23.459273,7.471770e-04,1.0
SIGLEC1,0.147784,0.031227,1.0,22.396751,1.154326e-03,1.0
CRTC2,0.182780,0.039729,1.0,21.166532,1.972636e-03,1.0
